# NFL Injury Rate Analysis: Data Cleaning and Joining

This notebook prepares three raw datasets for the main injury analysis. By the end, we will have a single clean table, `sports_clean.nfl_master`, with one row per player per game. Every row carries the player's attributes (position, experience, team), the game's context (surface type, weather, roof), and the player's injury status for that week.

**Source tables:**
- `sports_raw.nfl_injuries` — weekly injury report rows, 2009-2024
- `sports_raw.nfl_schedules` — one row per game with surface, weather, and roof data
- `sports_raw.nfl_rosters` — weekly roster with player attributes and experience

**End state:** `sports_clean.nfl_master` — analysis-ready, one row per player per game

## 1. Database Connection

The same connection pattern used in all project scripts. We manually read the `.env` file rather than using `python-dotenv` to keep dependencies minimal.

In [1]:
import os
import pg8000
import pandas as pd

# pandas is pinned to 1.3.4 across this project
print(f"pandas version: {pd.__version__}")

pandas version: 3.0.0


In [2]:
def load_env(path: str) -> None:
    """Read .env file into os.environ without overwriting existing vars."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())


def get_conn():
    """Return a pg8000 connection using values loaded from .env."""
    return pg8000.connect(
        host=os.environ["DB_HOST"],
        port=int(os.environ["DB_PORT"]),
        database=os.environ["DB_NAME"],
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
    )


load_env("L:/data_projects/projects/projects/nfl/.env")
conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT current_database(), current_user;")
print(cur.fetchone())
cur.close()

['projects_db', 'projects_user']


## 2. Explore Raw Data

Before cleaning anything, we want to know what we are working with: how many rows are in each table and what does a sample payload look like. This step catches ingestion issues early, such as wrong row counts, empty tables, or malformed JSON.

In [3]:
RAW_TABLES = [
    "sports_raw.nfl_injuries",
    "sports_raw.nfl_schedules",
    "sports_raw.nfl_rosters",
]

for table in RAW_TABLES:
    cur = conn.cursor()
    cur.execute(f"SELECT dataset_id, COUNT(*) FROM {table} GROUP BY dataset_id;")
    rows = cur.fetchall()
    cur.close()
    print(f"\n{table}:")
    for dataset_id, count in rows:
        print(f"  {dataset_id}: {count:,} rows")


sports_raw.nfl_injuries:
  injuries_2009_2024: 84,684 rows

sports_raw.nfl_schedules:
  schedules_2009_2024: 4,345 rows

sports_raw.nfl_rosters:
  rosters_2009_2024: 42,823 rows


## 3. Inspect Payload Structures

Each raw table stores records as JSONB objects. Unlike the F1 data (which had nested objects inside each payload), nflreadpy returns flat DataFrames, so our payloads contain only top-level key-value pairs. We still need to see the exact field names before writing our rename maps.

In [4]:
for table in RAW_TABLES:
    cur = conn.cursor()
    cur.execute(f"SELECT payload FROM {table} LIMIT 1;")
    row = cur.fetchone()
    cur.close()
    if row:
        keys = list(row[0].keys())
        print(f"\n{'='*60}")
        print(f"    {table.upper()}")
        print(f"{'='*60}")
        print(f"    Fields ({len(keys)}): {keys}")
    else:
        print(f"\n{table}: no rows found")


    SPORTS_RAW.NFL_INJURIES
    Fields (16): ['team', 'week', 'season', 'gsis_id', 'position', 'full_name', 'game_type', 'last_name', 'first_name', 'date_modified', 'report_status', 'practice_status', 'report_primary_injury', 'practice_primary_injury', 'report_secondary_injury', 'practice_secondary_injury']

    SPORTS_RAW.NFL_SCHEDULES
    Fields (46): ['ftn', 'pff', 'pfr', 'espn', 'gsis', 'roof', 'temp', 'week', 'wind', 'total', 'result', 'season', 'game_id', 'gameday', 'referee', 'stadium', 'surface', 'weekday', 'div_game', 'gametime', 'location', 'overtime', 'away_rest', 'away_team', 'game_type', 'home_rest', 'home_team', 'over_odds', 'away_coach', 'away_qb_id', 'away_score', 'home_coach', 'home_qb_id', 'home_score', 'stadium_id', 'total_line', 'under_odds', 'old_game_id', 'spread_line', 'away_qb_name', 'home_qb_name', 'nfl_detail_id', 'away_moneyline', 'home_moneyline', 'away_spread_odds', 'home_spread_odds']

    SPORTS_RAW.NFL_ROSTERS
    Fields (36): ['team', 'week', 'esb_id',

## 4. Clean the Injuries Table

The injury report is the core of our analysis. Each row represents one player's status on one week's injury report for a given team. We need to:

1. Pull all rows from `sports_raw.nfl_injuries` and flatten the JSONB payloads
2. Rename fields to clean snake_case
3. Cast types (season and week to int, dates to datetime)
4. Add an `is_injured` flag, our binary outcome variable

### Injury proxy methodology

We use `Out` and `IR` as our injury proxy. Players listed as `Questionable` who suit up and play are not truly absent from the game. `Out` and `IR` are the most consistently reported and clearly defined statuses across all seasons, and represent the cleanest signal that a player missed game action due to injury. This is a documented limitation: players who play through pain are undercounted.

In [5]:
query = """
    SELECT payload
    FROM sports_raw.nfl_injuries;
"""
df_inj_raw = pd.read_sql(query, conn)
print(f"Raw injury rows: {len(df_inj_raw):,}")

# pd.json_normalize converts a list of flat dicts into a DataFrame.
# Since nflreadpy payloads are flat (no nested objects), each field becomes
# a column directly with no dot-notation renaming needed.
df_injuries = pd.json_normalize(df_inj_raw["payload"])
print(f"Columns ({len(df_injuries.columns)}): {df_injuries.columns.tolist()}")
print(f"\nSample row:")
print(df_injuries.iloc[0])

C:\Users\nsumn\AppData\Local\Temp\ipykernel_10092\314687951.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_inj_raw = pd.read_sql(query, conn)


Raw injury rows: 84,684
Columns (16): ['team', 'week', 'season', 'gsis_id', 'position', 'full_name', 'game_type', 'last_name', 'first_name', 'date_modified', 'report_status', 'practice_status', 'report_primary_injury', 'practice_primary_injury', 'report_secondary_injury', 'practice_secondary_injury']

Sample row:
team                                                       ARI
week                                                       1.0
season                                                  2009.0
gsis_id                                             00-0022084
position                                                    WR
full_name                                        Anquan Boldin
game_type                                                  REG
last_name                                               Boldin
first_name                                              Anquan
date_modified                                              NaN
report_status                                     Quest

In [6]:
# gsis_id is the NFL's internal player identifier. It is consistent across
# injuries, schedules, and rosters, making it the reliable join key.
INJ_KEEP = {
    "season": "season",
    "week": "week",
    "team": "team",
    "full_name": "full_name",
    "gsis_id": "gsis_id",
    "position": "position",
    "report_status": "report_status",
    "report_primary_injury": "report_primary_injury",
    "report_secondary_injury": "report_secondary_injury",
    "practice_status": "practice_status",
    "game_type": "game_type",
    "date_modified": "date_modified",
}

# Guard against column names changing between nflverse releases
existing = {k: v for k, v in INJ_KEEP.items() if k in df_injuries.columns}
missing = [k for k in INJ_KEEP if k not in df_injuries.columns]
if missing:
    print(f"WARNING: expected columns not found: {missing}")

df_injuries = df_injuries[list(existing.keys())].rename(columns=existing)

df_injuries["season"] = pd.to_numeric(df_injuries["season"], errors="coerce")
df_injuries["week"] = pd.to_numeric(df_injuries["week"], errors="coerce")
df_injuries["date_modified"] = pd.to_datetime(df_injuries["date_modified"], errors="coerce")

print(f"Shape: {df_injuries.shape}")
print(df_injuries.dtypes)

Shape: (84684, 12)
season                                 float64
week                                   float64
team                                       str
full_name                                  str
gsis_id                                    str
position                                   str
report_status                              str
report_primary_injury                      str
report_secondary_injury                    str
practice_status                            str
game_type                                  str
date_modified              datetime64[us, UTC]
dtype: object


In [7]:
# See all report_status values before applying the proxy
print("Report status distribution:")
print(df_injuries["report_status"].value_counts(dropna=False))

# Out and IR are the injury proxy (see methodology note above)
INJURY_STATUSES = {"Out", "IR"}
df_injuries["is_injured"] = df_injuries["report_status"].isin(INJURY_STATUSES)

print(f"\nInjury proxy (Out or IR): {df_injuries['is_injured'].sum():,} of {len(df_injuries):,} rows")
print(f"Overall injury rate: {df_injuries['is_injured'].mean():.1%}")

Report status distribution:
report_status
NaN             26960
Questionable    22333
Probable        17400
Out             14725
Doubtful         3260
Note                6
Name: count, dtype: int64

Injury proxy (Out or IR): 14,725 of 84,684 rows
Overall injury rate: 17.4%


## 5. Clean the Schedules Table

Each row in the schedules table is one game. We need this data to attach surface type, weather conditions, and roof status to each player's injury record.

The critical wrinkle: schedules have one row per game with both `home_team` and `away_team`. Injuries have one row per player, meaning one row per team per week. To join them cleanly, we reshape schedules into a per-team format so each game appears twice: once for the home team and once for the away team. After the reshape, the join on `(season, week, team)` is straightforward.

In [8]:
query = """
    SELECT payload
    FROM sports_raw.nfl_schedules;
"""
df_sch_raw = pd.read_sql(query, conn)
print(f"Raw schedule rows: {len(df_sch_raw):,}")

df_schedules = pd.json_normalize(df_sch_raw["payload"])
print(f"Columns ({len(df_schedules.columns)}): {df_schedules.columns.tolist()}")

Raw schedule rows: 4,345
Columns (46): ['ftn', 'pff', 'pfr', 'espn', 'gsis', 'roof', 'temp', 'week', 'wind', 'total', 'result', 'season', 'game_id', 'gameday', 'referee', 'stadium', 'surface', 'weekday', 'div_game', 'gametime', 'location', 'overtime', 'away_rest', 'away_team', 'game_type', 'home_rest', 'home_team', 'over_odds', 'away_coach', 'away_qb_id', 'away_score', 'home_coach', 'home_qb_id', 'home_score', 'stadium_id', 'total_line', 'under_odds', 'old_game_id', 'spread_line', 'away_qb_name', 'home_qb_name', 'nfl_detail_id', 'away_moneyline', 'home_moneyline', 'away_spread_odds', 'home_spread_odds']


C:\Users\nsumn\AppData\Local\Temp\ipykernel_10092\2992753536.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sch_raw = pd.read_sql(query, conn)


In [9]:
SCH_KEEP = {
    "game_id": "game_id",
    "season": "season",
    "week": "week",
    "game_type": "game_type",
    "gameday": "gameday",
    "gametime": "gametime",
    "home_team": "home_team",
    "away_team": "away_team",
    "home_score": "home_score",
    "away_score": "away_score",
    "surface": "surface",
    "roof": "roof",
    "temp": "temp",
    "wind": "wind",
    "stadium": "stadium",
    "div_game": "div_game",
    "overtime": "overtime",
}

existing_sch = {k: v for k, v in SCH_KEEP.items() if k in df_schedules.columns}
missing_sch = [k for k in SCH_KEEP if k not in df_schedules.columns]
if missing_sch:
    print(f"WARNING: expected columns not found: {missing_sch}")

df_schedules = df_schedules[list(existing_sch.keys())].rename(columns=existing_sch)

df_schedules["season"] = pd.to_numeric(df_schedules["season"], errors="coerce")
df_schedules["week"] = pd.to_numeric(df_schedules["week"], errors="coerce")
df_schedules["home_score"] = pd.to_numeric(df_schedules["home_score"], errors="coerce")
df_schedules["away_score"] = pd.to_numeric(df_schedules["away_score"], errors="coerce")
df_schedules["temp"] = pd.to_numeric(df_schedules["temp"], errors="coerce")
df_schedules["wind"] = pd.to_numeric(df_schedules["wind"], errors="coerce")
df_schedules["gameday"] = pd.to_datetime(df_schedules["gameday"], errors="coerce")

print(f"Shape: {df_schedules.shape}")
print(df_schedules.dtypes)

Shape: (4345, 17)
game_id                  str
season                 int64
week                   int64
game_type                str
gameday       datetime64[us]
gametime                 str
home_team                str
away_team                str
home_score             int64
away_score             int64
surface                  str
roof                     str
temp                 float64
wind                 float64
stadium                  str
div_game               int64
overtime               int64
dtype: object


In [10]:
# Surface names in nflverse are inconsistent: 'FieldTurf', 'fieldturf', and
# 'Field Turf' all refer to the same artificial surface.
# We normalize everything to 'artificial' or 'natural'.
print("Raw surface values:")
print(df_schedules["surface"].value_counts(dropna=False))

Raw surface values:
surface
grass         2333
fieldturf     1138
sportturf      262
matrixturf     195
astroturf      109
a_turf         101
grass           93
astroplay       63
                43
dessograss       8
Name: count, dtype: int64


In [ ]:
# Lowercase first so matching is case-insensitive
df_schedules["surface_raw"] = df_schedules["surface"].str.lower().str.strip()

SURFACE_MAP = {
    # Artificial surfaces
    "fieldturf": "artificial",
    "field turf": "artificial",
    "matrixturf": "artificial",
    "matrix turf": "artificial",
    "hellas matrix": "artificial",
    "sprinturf": "artificial",
    "a_turf": "artificial",
    "a turf": "artificial",
    "astroturf": "artificial",
    "astro turf": "artificial",
    "astroplay": "artificial",
    "artificialturn": "artificial",  # known typo in nflverse data
    "fieldturf360": "artificial",
    "mondo": "artificial",
    "dessograss": "artificial",
    "sportexe momentum": "artificial",
    "sportexe": "artificial",
    "sportturf": "artificial",
    # Natural surfaces
    "grass": "natural",
    "bermuda grass": "natural",
    "bluegrass": "natural",
    "kentucky bluegrass": "natural",
    "naturaturf": "natural",
    "tiftuf bermudagrass": "natural",
    "natural grass": "natural",
}

df_schedules["surface_type"] = df_schedules["surface_raw"].map(SURFACE_MAP).fillna("unknown")

# Any 'unknown' values need to be added to SURFACE_MAP above
unknowns = df_schedules[df_schedules["surface_type"] == "unknown"]["surface_raw"].value_counts()
if len(unknowns) > 0:
    print(f"Unmapped surface values (add to SURFACE_MAP):")
    print(unknowns)
else:
    print("All surface values mapped successfully.")

print(f"\nStandardized surface distribution:")
print(df_schedules["surface_type"].value_counts())

Unmapped surface values (add to SURFACE_MAP):
surface_raw
sportturf    262
              43
Name: count, dtype: int64

Standardized surface distribution:
surface_type
natural       2426
artificial    1614
unknown        305
Name: count, dtype: int64


## 5b. Reshape Schedules from Per-Game to Per-Team

Schedules have one row per game. Injuries have one row per player, meaning one row per team per week. To join them, we reshape schedules so each game appears twice: once for the home team and once for the away team. After this step, we can join on `(season, week, team)`.

In [12]:
GAME_COLS = [
    "game_id", "season", "week", "game_type", "gameday", "gametime",
    "home_score", "away_score", "surface_type", "roof", "temp",
    "wind", "stadium", "div_game", "overtime",
]
# Only keep columns that actually exist after cleaning
GAME_COLS = [c for c in GAME_COLS if c in df_schedules.columns]

# Home side: this team is at home, the opponent is the away team
home = df_schedules[GAME_COLS + ["home_team", "away_team"]].copy()
home = home.rename(columns={"home_team": "team", "away_team": "opponent"})
home["is_home"] = True
home["team_score"] = home["home_score"]
home["opp_score"] = home["away_score"]

# Away side: this team is traveling, the opponent is the home team
away = df_schedules[GAME_COLS + ["home_team", "away_team"]].copy()
away = away.rename(columns={"away_team": "team", "home_team": "opponent"})
away["is_home"] = False
away["team_score"] = away["away_score"]
away["opp_score"] = away["home_score"]

df_games = pd.concat([home, away], ignore_index=True)
df_games = df_games.drop(columns=["home_score", "away_score"])

print(f"Schedules reshaped: {len(df_schedules):,} games -> {len(df_games):,} per-team rows")
print(df_games.head(3))

Schedules reshaped: 4,345 games -> 8,690 per-team rows
           game_id  season  week game_type    gameday gametime surface_type  \
0  2009_01_TEN_PIT    2009     1       REG 2009-09-10    20:30   artificial   
1  2009_01_MIA_ATL    2009     1       REG 2009-09-13    13:00   artificial   
2   2009_01_KC_BAL    2009     1       REG 2009-09-13    13:00      unknown   

       roof  temp  wind           stadium  div_game  overtime team opponent  \
0  outdoors  67.0   9.0       Heinz Field         0         1  PIT      TEN   
1      dome   NaN   NaN      Georgia Dome         0         0  ATL      MIA   
2  outdoors  76.0   5.0  M&T Bank Stadium         0         0  BAL       KC   

   is_home  team_score  opp_score  
0     True          13         10  
1     True          19          7  
2     True          38         24  


## 6. Clean the Rosters Table

The roster table gives us player attributes: position, years of experience, height, weight, and draft information. We use this to answer two key questions:

1. Does injury rate differ by position (linemen vs. skill positions vs. quarterbacks)?
2. Are rookies injured at a higher rate than veterans?

### Rookie identification

We define a rookie as a player in their first NFL season. The `years_exp == 0` flag catches most cases, but undrafted free agents who were cut and re-signed can have misleading `years_exp` values. We use `entry_year == season` as a cross-check so undrafted rookies are not missed.

In [13]:
query = """
    SELECT payload
    FROM sports_raw.nfl_rosters;
"""
df_ros_raw = pd.read_sql(query, conn)
print(f"Raw roster rows: {len(df_ros_raw):,}")

df_rosters = pd.json_normalize(df_ros_raw["payload"])
print(f"Columns ({len(df_rosters.columns)}): {df_rosters.columns.tolist()}")

C:\Users\nsumn\AppData\Local\Temp\ipykernel_10092\193284638.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ros_raw = pd.read_sql(query, conn)


Raw roster rows: 42,823
Columns (36): ['team', 'week', 'esb_id', 'height', 'pff_id', 'pfr_id', 'season', 'status', 'weight', 'college', 'espn_id', 'gsis_id', 'position', 'smart_id', 'yahoo_id', 'full_name', 'game_type', 'last_name', 'years_exp', 'birth_date', 'draft_club', 'entry_year', 'first_name', 'gsis_it_id', 'sleeper_id', 'rookie_year', 'rotowire_id', 'draft_number', 'headshot_url', 'ngs_position', 'football_name', 'jersey_number', 'sportradar_id', 'fantasy_data_id', 'depth_chart_position', 'status_description_abbr']


In [14]:
ROS_KEEP = {
    "season": "season",
    "team": "team",
    "full_name": "full_name",
    "gsis_id": "gsis_id",
    "position": "position",
    "depth_chart_position": "depth_chart_position",
    "jersey_number": "jersey_number",
    "height": "height",
    "weight": "weight",
    "birth_date": "birth_date",
    "college": "college",
    "years_exp": "years_exp",
    "entry_year": "entry_year",
    "rookie_year": "rookie_year",
    "draft_club": "draft_club",
    "draft_number": "draft_number",
    "status": "roster_status",
}

existing_ros = {k: v for k, v in ROS_KEEP.items() if k in df_rosters.columns}
missing_ros = [k for k in ROS_KEEP if k not in df_rosters.columns]
if missing_ros:
    print(f"WARNING: expected columns not found: {missing_ros}")

df_rosters = df_rosters[list(existing_ros.keys())].rename(columns=existing_ros)

df_rosters["season"] = pd.to_numeric(df_rosters["season"], errors="coerce")
df_rosters["years_exp"] = pd.to_numeric(df_rosters["years_exp"], errors="coerce")
df_rosters["entry_year"] = pd.to_numeric(df_rosters["entry_year"], errors="coerce")
df_rosters["rookie_year"] = pd.to_numeric(df_rosters["rookie_year"], errors="coerce")
df_rosters["draft_number"] = pd.to_numeric(df_rosters["draft_number"], errors="coerce")
df_rosters["height"] = pd.to_numeric(df_rosters["height"], errors="coerce")
df_rosters["weight"] = pd.to_numeric(df_rosters["weight"], errors="coerce")
df_rosters["birth_date"] = pd.to_datetime(df_rosters["birth_date"], errors="coerce")

print(f"Shape: {df_rosters.shape}")
print(df_rosters.dtypes)

Shape: (42823, 17)
season                           int64
team                               str
full_name                          str
gsis_id                            str
position                           str
depth_chart_position               str
jersey_number                      str
height                         float64
weight                         float64
birth_date              datetime64[us]
college                            str
years_exp                      float64
entry_year                     float64
rookie_year                    float64
draft_club                         str
draft_number                   float64
roster_status                      str
dtype: object


In [15]:
# Roster data can have one row per player per week.
# For joining to injuries, we want one row per player per season per team.
# We sort by season and keep the last entry for each (season, team, gsis_id).

# is_rookie: primary signal is years_exp == 0,
# cross-check with entry_year == season to catch undrafted free agents
df_rosters["is_rookie"] = (
    (df_rosters["years_exp"] == 0)
    | (df_rosters["entry_year"] == df_rosters["season"])
)

df_rosters_dedup = (
    df_rosters
    .sort_values("season")
    .drop_duplicates(subset=["season", "team", "gsis_id"], keep="last")
    .reset_index(drop=True)
)

print(f"Before dedup: {len(df_rosters):,}")
print(f"After dedup:  {len(df_rosters_dedup):,}")
print(f"\nRookie count by season (last 10):")
print(df_rosters_dedup.groupby("season")["is_rookie"].sum().tail(10))

Before dedup: 42,823
After dedup:  42,816

Rookie count by season (last 10):
season
2015    358
2016    542
2017    777
2018    786
2019    766
2020    594
2021    487
2022    707
2023    671
2024    705
Name: is_rookie, dtype: int64


## 7. Build nfl_master: Joining the Three Datasets

The master table is the unit of observation for the logistic regression. Each row is one player's appearance on a weekly injury report, enriched with game context and player attributes.

**Join sequence:**
1. Start with `df_injuries`, one row per player per week
2. Join `df_games` (per-team game context) on `(season, week, team)` to add surface, weather, and roof
3. Join `df_rosters_dedup` (player attributes) on `(season, team, gsis_id)` to add position and experience

Players on the injury report with no matching game (bye weeks, preseason oddities) are dropped by the inner join. Players with no roster match get null attribute columns, which we flag for review.

In [16]:
# Step 1: attach game context to each injury row
df_master = df_injuries.merge(
    df_games,
    on=["season", "week", "team"],
    how="inner",
)

print(f"Injuries before join: {len(df_injuries):,}")
print(f"After joining games:  {len(df_master):,}")
print(f"Dropped:              {len(df_injuries) - len(df_master):,}")

Injuries before join: 84,684
After joining games:  84,667
Dropped:              17


In [17]:
# Step 2: attach player attributes from rosters
ROSTER_COLS = [
    "season", "team", "gsis_id",
    "position", "depth_chart_position", "height", "weight",
    "birth_date", "college", "years_exp", "entry_year",
    "draft_number", "roster_status", "is_rookie",
]
ROSTER_COLS = [c for c in ROSTER_COLS if c in df_rosters_dedup.columns]

df_master = df_master.merge(
    df_rosters_dedup[ROSTER_COLS],
    on=["season", "team", "gsis_id"],
    how="left",
)

no_roster = df_master["years_exp"].isna().sum()
print(f"Rows with no roster match: {no_roster:,} ({no_roster/len(df_master):.1%})")

# When both DataFrames have a 'position' column, pandas suffixes them _x and _y.
# Roster position uses a more standardized taxonomy so we prefer it,
# falling back to the injury report position only when the roster value is missing.
if "position_x" in df_master.columns and "position_y" in df_master.columns:
    df_master["position"] = df_master["position_y"].combine_first(df_master["position_x"])
    df_master = df_master.drop(columns=["position_x", "position_y"])

# Same dedup for game_type if it appears in both injuries and schedules
if "game_type_x" in df_master.columns and "game_type_y" in df_master.columns:
    df_master["game_type"] = df_master["game_type_x"].combine_first(df_master["game_type_y"])
    df_master = df_master.drop(columns=["game_type_x", "game_type_y"])

print(f"\nFinal master shape: {df_master.shape}")

Rows with no roster match: 7,219 (8.5%)

Final master shape: (84667, 37)


In [18]:
# Define final column order: identifiers first, then player attributes,
# then game context, then the outcome variable last
COL_ORDER = [
    # Game identifiers
    "game_id", "season", "week", "game_type", "team", "opponent",
    # Player identifiers
    "gsis_id", "full_name",
    # Player attributes
    "position", "depth_chart_position", "height", "weight",
    "birth_date", "college", "years_exp", "entry_year",
    "draft_number", "is_rookie", "roster_status",
    # Game context
    "gameday", "gametime", "is_home", "team_score", "opp_score",
    "surface_type", "roof", "temp", "wind", "stadium", "div_game", "overtime",
    # Injury outcome
    "report_status", "report_primary_injury", "report_secondary_injury",
    "practice_status", "date_modified", "is_injured",
]

COL_ORDER = [c for c in COL_ORDER if c in df_master.columns]
df_master = df_master[COL_ORDER]

print(f"Final shape: {df_master.shape}")
print(f"\nNull counts by column:")
nulls = df_master.isnull().sum()
print(nulls[nulls > 0])

# Quick sanity check: injury rate by surface
if "surface_type" in df_master.columns:
    print(f"\nInjury rate by surface type:")
    print(df_master.groupby("surface_type")["is_injured"].mean().round(4))

Final shape: (84667, 37)

Null counts by column:
depth_chart_position       36130
height                      7023
weight                      7023
birth_date                  7024
college                    19444
years_exp                   7219
entry_year                  7219
draft_number               25841
is_rookie                   7023
roster_status               7023
temp                       26584
wind                       26584
report_status              26945
report_primary_injury      26951
report_secondary_injury    81465
date_modified               4866
dtype: int64

Injury rate by surface type:
surface_type
artificial    0.1716
natural       0.1744
unknown       0.1835
Name: is_injured, dtype: float64


## 8. Save nfl_master to sports_clean

We write the final joined DataFrame to `sports_clean.nfl_master`. This is the table the analysis notebook will read from directly. The same pg8000 batch insert pattern used across all project scripts handles the write.

In [19]:
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS sports_clean;")
conn.commit()
cur.execute("DROP TABLE IF EXISTS sports_clean.nfl_master;")
conn.commit()

# Build the CREATE TABLE statement from COL_ORDER.
# Types match what we cast earlier in this notebook.
CREATE_SQL = """
CREATE TABLE sports_clean.nfl_master (
    game_id                 TEXT,
    season                  INT,
    week                    INT,
    game_type               TEXT,
    team                    TEXT,
    opponent                TEXT,
    gsis_id                 TEXT,
    full_name               TEXT,
    position                TEXT,
    depth_chart_position    TEXT,
    height                  FLOAT,
    weight                  FLOAT,
    birth_date              DATE,
    college                 TEXT,
    years_exp               FLOAT,
    entry_year              FLOAT,
    draft_number            FLOAT,
    is_rookie               BOOLEAN,
    roster_status           TEXT,
    gameday                 DATE,
    gametime                TEXT,
    is_home                 BOOLEAN,
    team_score              FLOAT,
    opp_score               FLOAT,
    surface_type            TEXT,
    roof                    TEXT,
    temp                    FLOAT,
    wind                    FLOAT,
    stadium                 TEXT,
    div_game                BOOLEAN,
    overtime                BOOLEAN,
    report_status           TEXT,
    report_primary_injury   TEXT,
    report_secondary_injury TEXT,
    practice_status         TEXT,
    date_modified           TIMESTAMPTZ,
    is_injured              BOOLEAN
);
"""

cur.execute(CREATE_SQL)
conn.commit()
cur.close()
print("Table sports_clean.nfl_master created.")

Table sports_clean.nfl_master created.


In [20]:
# The number of %s placeholders must match the number of columns in df_master
n_cols = len(df_master.columns)
placeholders = ",".join(["%s"] * n_cols)
INSERT_SQL = f"INSERT INTO sports_clean.nfl_master VALUES ({placeholders})"


def safe_val(v):
    """Convert NaN/NaT to None and float whole numbers to int for pg8000."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    # pandas stores int columns as float64 when NaNs are present (e.g. 2009.0).
    # PostgreSQL INT columns reject "2009.0", so we convert whole-number floats to int.
    if isinstance(v, float) and v.is_integer():
        return int(v)
    return v


BATCH_SIZE = 5000
cur = conn.cursor()
batch = []
n = 0

for _, row in df_master.iterrows():
    batch.append(tuple(safe_val(v) for v in row.values))
    if len(batch) >= BATCH_SIZE:
        cur.executemany(INSERT_SQL, batch)
        conn.commit()
        n += len(batch)
        print(f"  Inserted {n:,} rows...")
        batch = []

if batch:
    cur.executemany(INSERT_SQL, batch)
    conn.commit()
    n += len(batch)

cur.close()
print(f"\nsports_clean.nfl_master: {n:,} rows saved")

  Inserted 5,000 rows...
  Inserted 10,000 rows...
  Inserted 15,000 rows...
  Inserted 20,000 rows...
  Inserted 25,000 rows...
  Inserted 30,000 rows...
  Inserted 35,000 rows...
  Inserted 40,000 rows...
  Inserted 45,000 rows...
  Inserted 50,000 rows...
  Inserted 55,000 rows...
  Inserted 60,000 rows...
  Inserted 65,000 rows...
  Inserted 70,000 rows...
  Inserted 75,000 rows...
  Inserted 80,000 rows...

sports_clean.nfl_master: 84,667 rows saved


## 9. Validation

A few sanity checks to confirm the data landed correctly: row counts match the DataFrame, the injury rate is plausible by surface type, and season coverage runs 2009-2024.

In [21]:
# Confirm DB row count matches DataFrame
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM sports_clean.nfl_master;")
db_count = cur.fetchone()[0]
cur.close()
print(f"Rows in DB:          {db_count:,}")
print(f"Rows in DataFrame:   {len(df_master):,}")
assert db_count == len(df_master), "Row count mismatch between DataFrame and DB"

# Injury rate by surface type (the key variable for our analysis)
cur = conn.cursor()
cur.execute("""
    SELECT
        surface_type,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_injured THEN 1 ELSE 0 END) AS injured,
        ROUND(AVG(CASE WHEN is_injured THEN 1.0 ELSE 0.0 END), 4) AS injury_rate
    FROM sports_clean.nfl_master
    GROUP BY surface_type
    ORDER BY injury_rate DESC;
""")
print("\nInjury rate by surface type:")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[2]:,} injured / {row[1]:,} total ({float(row[3]):.1%})")
cur.close()

# Season coverage
cur = conn.cursor()
cur.execute("""
    SELECT season, COUNT(*) AS rows,
           SUM(CASE WHEN is_injured THEN 1 ELSE 0 END) AS injured
    FROM sports_clean.nfl_master
    GROUP BY season
    ORDER BY season;
""")
print("\nRows and injuries by season:")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]:,} rows, {row[2]:,} injured")
cur.close()

Rows in DB:          84,667
Rows in DataFrame:   84,667

Injury rate by surface type:
  unknown: 973 injured / 5,303 total (18.4%)
  natural: 8,257 injured / 47,342 total (17.4%)
  artificial: 5,495 injured / 32,022 total (17.2%)

Rows and injuries by season:
  2009: 4,821 rows, 709 injured
  2010: 4,491 rows, 751 injured
  2011: 4,971 rows, 877 injured
  2012: 5,533 rows, 836 injured
  2013: 5,070 rows, 785 injured
  2014: 5,078 rows, 913 injured
  2015: 5,232 rows, 979 injured
  2016: 5,115 rows, 1,043 injured
  2017: 5,104 rows, 897 injured
  2018: 5,133 rows, 920 injured
  2019: 5,392 rows, 1,036 injured
  2020: 5,661 rows, 901 injured
  2021: 5,587 rows, 888 injured
  2022: 5,665 rows, 1,078 injured
  2023: 5,599 rows, 996 injured
  2024: 6,215 rows, 1,116 injured
